In [8]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [9]:
df = pd.read_csv("train.csv")

# Separate label and pixels
y = df["label"].values
X = df.drop(columns=["label"]).values

# Normalize pixels from 0-255 -> 0-1
X = X.astype("float32") / 255.0

# Reshape: (number of images, 784) -> (number of images, 28, 28)
X_image = X.reshape(-1, 28, 28)

print(X_image.shape)
print(y.shape)

(42000, 28, 28)
(42000,)


In [10]:
df_ = pd.read_csv("test.csv")

# Separate label and pixels
y_test = df["label"].values
X_test = df.drop(columns=["label"]).values

# Normalize pixels from 0-255 -> 0-1
X_test = X_test.astype("float32") / 255.0

print(X_test.shape)
print(y_test.shape)

(42000, 784)
(42000,)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_val.shape)

(33600, 784)
(8400, 784)


In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    solver="adam",
    max_iter=200,
    random_state=42
)

param_grid = {
    "hidden_layer_sizes": [(64,), (128,), (128, 64)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001],
    "learning_rate_init": [0.001, 0.0005]
}

grid = GridSearchCV(
    mlp,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

Best parameters: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (128, 64), 'learning_rate_init': 0.001}
Best CV accuracy: 0.9709523809523809


In [21]:
best_mlp = grid.best_estimator_

y_pred = best_mlp.predict(X_val)

In [22]:
from sklearn.metrics import accuracy_score, classification_report

print("MLP Accuracy:", accuracy_score(y_val, y_pred))

print("\nMLP Classification Report:")
print(classification_report(y_val, y_pred))

MLP Accuracy: 0.9746428571428571

MLP Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       827
           1       0.98      0.99      0.99       937
           2       0.98      0.97      0.98       835
           3       0.97      0.95      0.96       870
           4       0.98      0.98      0.98       814
           5       0.96      0.97      0.97       759
           6       0.99      0.99      0.99       827
           7       0.98      0.98      0.98       880
           8       0.98      0.97      0.97       813
           9       0.95      0.96      0.96       838

    accuracy                           0.97      8400
   macro avg       0.97      0.97      0.97      8400
weighted avg       0.97      0.97      0.97      8400



In [23]:
# Save the model parameter
joblib.dump(best_mlp, "mlp_model.pkl")

['mlp_model.pkl']

In [24]:
mlp = joblib.load("mlp_model.pkl")
prediction = mlp.predict(X)

print("Predictions:", prediction)

Predictions: [1 0 1 ... 7 6 9]
